# 第 14 章实验：端到端推理、导出与一致性
把合成 RGB 输入、预处理、模型输出、TorchScript 跟踪、torch.export 导出和延迟统计串成最小离线流水线。导出文件只存在于临时目录。

In [ ]:
import io
import tempfile
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 图里的中文需要中文字体：Windows 的字体放前面，后面是 macOS 和 Linux 的
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 有 NVIDIA 显卡就用显卡，没有就用 CPU。这个实验很小，在 CPU 上也只要几秒
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

torch.manual_seed(14)
input_hw = (32, 32)
class_names = ['empty', 'object']

In [ ]:
image = np.zeros((60, 100, 3), dtype=np.float32)
image[15:48, 35:75, 0] = 0.9
image[15:48, 35:75, 1] = 0.35

def preprocess(rgb_hwc):
    assert rgb_hwc.ndim == 3 and rgb_hwc.shape[2] == 3
    tensor = torch.from_numpy(rgb_hwc).permute(2, 0, 1).unsqueeze(0)
    return F.interpolate(tensor, size=input_hw, mode='bilinear', align_corners=False).to(device)

x = preprocess(image)
print('original HWC:', image.shape, 'model input NCHW:', tuple(x.shape), x.dtype)

In [ ]:
class DeployableTinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3, 6, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(6, 2))
    def forward(self, x):
        return self.net(x)

model = DeployableTinyNet().to(device).eval()
with torch.inference_mode():
    logits = model(x)
    probabilities = logits.softmax(1)
result = {'class_id': int(probabilities.argmax(1)), 'score': float(probabilities.max())}
print('output logits:', tuple(logits.shape), 'result contract:', result)

## 导出一：TorchScript 跟踪
`torch.jit.trace` 拿一个样例输入把模型跑一遍，记下执行过的运算。在较新的 PyTorch 上，`torch.jit.trace` 和 `torch.jit.load` 会打印弃用警告（FutureWarning），建议改用 `torch.export` 或 `torch.compile`。警告不影响这个单元的结果。这里保留它，是因为讲义 14.8 节关于“跟踪会把分支固定下来”的实测用的就是它；下一个单元用 `torch.export` 把同一个模型再导出一次。

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'tiny_model.pt'
    traced = torch.jit.trace(model, x)
    traced.save(str(path))
    loaded = torch.jit.load(str(path), map_location=device).eval()
    with torch.inference_mode():
        eager_out = model(x)
        exported_out = loaded(x)
    max_error = float((eager_out - exported_out).abs().max())
    file_size = path.stat().st_size
print(f'export max abs error={max_error:.3e}, temporary file bytes={file_size}')
assert max_error < 1e-6

## 导出二：torch.export（当前推荐的方式）
`torch.export.export` 同样要一个样例输入，得到一个 `ExportedProgram`；`torch.export.save` 把它存起来（这里存进内存，传文件路径就是存成 `.pt2` 文件），`torch.export.load` 读回来，再用 `.module()` 得到可以调用的模块。和跟踪相比有两点不同：模型里有随输入内容变化的分支时它直接报错，不会悄悄固定一条分支（讲义 14.8 节）；它默认把样例输入的形状也固定下来，下面最后几行换一个批大小试一次。要支持可变的批大小，导出时用 `dynamic_shapes` 参数声明，写法以你安装的 PyTorch 版本的文档为准。

In [ ]:
if not hasattr(torch, 'export'):
    print('这个版本的 PyTorch 没有 torch.export，跳过这个单元。')
else:
    program = torch.export.export(model, (x,))
    buffer = io.BytesIO()
    torch.export.save(program, buffer)
    export_bytes = buffer.getbuffer().nbytes
    buffer.seek(0)
    reloaded = torch.export.load(buffer).module()
    with torch.inference_mode():
        export_error = float((model(x) - reloaded(x)).abs().max())
    print(f'torch.export max abs error={export_error:.3e}, saved bytes={export_bytes}')
    assert export_error < 1e-6
    try:
        with torch.inference_mode():
            reloaded(torch.cat([x, x], dim=0))
        print('批大小 2 的输入：被接受')
    except Exception as exc:
        print(f'批大小 2 的输入：被拒绝（{type(exc).__name__}）。导出时样例的批大小是 1，形状被固定了。')

In [ ]:
def benchmark(module, tensor, repeats=40):
    def wait():
        # 显卡上的运算是异步提交的，计时的两头都要等它算完；在 CPU 上什么也不做
        if tensor.is_cuda: torch.cuda.synchronize()
    samples_ms = []
    with torch.inference_mode():
        for _ in range(5): module(tensor)
        wait()
        for _ in range(repeats):
            start = time.perf_counter(); module(tensor); wait(); samples_ms.append((time.perf_counter()-start)*1000)
    return float(np.percentile(samples_ms, 50)), float(np.percentile(samples_ms, 95))

p50, p95 = benchmark(model, x)
print(f'{device.type} single-sample latency: P50={p50:.3f} ms, P95={p95:.3f} ms')
print('这是教学测量，不是目标设备正式基准。')

## 破坏实验：RGB/BGR 契约不一致
部署前后颜色通道顺序必须一致。下面交换通道并比较 logits。

In [ ]:
bgr_x = preprocess(image[..., ::-1].copy())
with torch.inference_mode(): channel_error = float((model(x) - model(bgr_x)).abs().max())
print(f'RGB/BGR swap changed logits by max {channel_error:.6f}')
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
axes[0].imshow(image); axes[0].set_title('RGB 输入')
axes[1].bar(class_names, probabilities[0].detach().cpu()); axes[1].set_ylim(0, 1); axes[1].set_title('输出概率')
axes[0].axis('off'); plt.tight_layout(); plt.show()

## 记录要求
正式项目还需固定数据版本、预处理、类别表、阈值和后处理，并在真实目标设备测端到端 P50/P95、峰值内存与任务指标。